[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Train YOLO11 Object Detection on a Custom Dataset

---

[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/ultralytics/ultralytics)

YOLO11 builds on the advancements introduced in YOLOv9 and YOLOv10 earlier this year, incorporating improved architectural designs, enhanced feature extraction techniques, and optimized training methods.

YOLO11m achieves a higher mean mAP score on the COCO dataset while using 22% fewer parameters than YOLOv8m, making it computationally lighter without sacrificing performance.

YOLOv11 is available in 5 different sizes, ranging from `2.6M` to `56.9M` parameters, and capable of achieving from `39.5` to `54.7` mAP on the COCO dataset.

## Setup

### Configure API keys

To fine-tune YOLO11, you need to provide your Roboflow API key. Follow these steps:

- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Colab, go to the left pane and click on `Secrets` (🔑). Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

### Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [1]:
!nvidia-smi

Wed Jun 17 15:37:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [2]:
import os
HOME = os.getcwd()
print(HOME)

/content


## Install YOLO11 via Ultralytics

In [3]:
%pip install "ultralytics<=8.3.40" supervision roboflow
# prevent ultralytics from tracking your activity
!yolo settings sync=False
import ultralytics
ultralytics.checks()

Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.5/112.6 GB disk)


## Inference with model pre-trained on COCO dataset

### CLI

**NOTE:** CLI requires no customization or Python code. You can simply run all tasks from the terminal with the yolo command.

In [4]:
!yolo task=detect mode=predict model=yolo11n.pt conf=0.25 source='https://media.roboflow.com/notebooks/examples/dog.jpeg' save=True

100% 5.35M/5.35M [00:00<00:00, 94.5MB/s]
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 238 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs

100% 104k/104k [00:00<00:00, 28.2MB/s]
image 1/1 /content/dog.jpeg: 640x384 2 persons, 1 car, 1 dog, 1 handbag, 120.5ms
Speed: 3.8ms preprocess, 120.5ms inference, 157.2ms postprocess per image at shape (1, 3, 640, 384)
Results saved to runs/detect/predict
💡 Learn more at https://docs.ultralytics.com/modes/predict


**NOTE:** Result annotated image got saved in `{HOME}/runs/detect/predict/`. Let's display it.

In [5]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/predict/dog.jpeg', width=600)

FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/predict/dog.jpeg'

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [8]:
%cd /content/drive/MyDrive/Package-4

[Errno 2] No such file or directory: '/content/drive/MyDrive/Package-4'
/content


In [10]:
import os

os.makedirs("/content/drive/MyDrive/Package-4", exist_ok=True)

In [11]:
import os

base = "/content/drive/MyDrive/Package-4"

os.makedirs(base + "/dataset/images/train", exist_ok=True)
os.makedirs(base + "/dataset/images/val", exist_ok=True)
os.makedirs(base + "/dataset/labels/train", exist_ok=True)
os.makedirs(base + "/dataset/labels/val", exist_ok=True)

In [12]:
yaml_content = """
path: /content/drive/MyDrive/Package-4/dataset
train: images/train
val: images/val

names:
  0: damaged
"""

with open("/content/drive/MyDrive/Package-4/dataset/data.yaml", "w") as f:
    f.write(yaml_content)

In [13]:
import os

base = "/content/drive/MyDrive/Package-4/dataset"

print(os.listdir(base))
print(os.listdir(base + "/images/train"))
print(os.listdir(base + "/labels/train"))

['images', 'labels', 'data.yaml']
[]
[]


In [14]:
!ls /content/drive/MyDrive/Package-4/dataset/images

train  val


In [16]:
!ls /content/drive/MyDrive/Package-4/dataset/images/train

### SDK

**NOTE:** YOLO's Python interface allows for seamless integration into your Python projects, making it easy to load, run, and process the model's output.

In [18]:
!ls /content/drive/MyDrive/Package-4/dataset/images/train

In [ ]:
from ultralytics import YOLO
from PIL import Image
import requests

model = YOLO('yolo11n.pt')
image = Image.open(requests.get('https://media.roboflow.com/notebooks/examples/dog.jpeg', stream=True).raw)
result = model.predict(image, conf=0.25)[0]

In [19]:
!ls /content/drive/MyDrive/Package-4/dataset/images/train

**NOTE:** The obtained `result` object stores information about the location, classes, and confidence levels of the detected objects.

In [ ]:
result.boxes.xyxy

In [ ]:
result.boxes.conf

In [ ]:
result.boxes.cls

In [20]:
!ls /content/drive/MyDrive/Package-4/dataset/labels/train

**NOTE:** YOLO11 can be easily integrated with `supervision` using the familiar `from_ultralytics` connector.

In [ ]:
import supervision as sv

detections = sv.Detections.from_ultralytics(result)

In [ ]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator(text_color=sv.Color.BLACK)

annotated_image = image.copy()
annotated_image = box_annotator.annotate(annotated_image, detections=detections)
annotated_image = label_annotator.annotate(annotated_image, detections=detections)

sv.plot_image(annotated_image, size=(10, 10))

## Fine-tune YOLO11 on custom dataset

**NOTE:** When training YOLOv11, make sure your data is located in `datasets`. If you'd like to change the default location of the data you want to use for fine-tuning, you can do so through Ultralytics' `settings.json`. In this tutorial, we will use one of the [datasets](https://universe.roboflow.com/liangdianzhong/-qvdww) available on [Roboflow Universe](https://universe.roboflow.com/). When downloading, make sure to select the `yolov11` export format.

In [21]:
!find /content/drive/MyDrive/Package-4/dataset -type f | head -20

/content/drive/MyDrive/Package-4/dataset/data.yaml


In [22]:
!find /content/drive/MyDrive/Package-4/dataset/images -type f | head -10

In [23]:
!ls -R /content/drive/MyDrive/Package-4/dataset

/content/drive/MyDrive/Package-4/dataset:
data.yaml  images  labels

/content/drive/MyDrive/Package-4/dataset/images:
train  val

/content/drive/MyDrive/Package-4/dataset/images/train:

/content/drive/MyDrive/Package-4/dataset/images/val:

/content/drive/MyDrive/Package-4/dataset/labels:
train  val

/content/drive/MyDrive/Package-4/dataset/labels/train:

/content/drive/MyDrive/Package-4/dataset/labels/val:


In [24]:
!find /content -name "data.yaml"

/content/drive/MyDrive/Package-4/dataset/data.yaml
/content/datasets/Packaging-4/data.yaml


In [25]:
!ls -R /content/datasets/Packaging-4 | head -50

/content/datasets/Packaging-4:
data.yaml
README.roboflow.txt
test
train
valid

/content/datasets/Packaging-4/test:
images
labels

/content/datasets/Packaging-4/test/images:
101_jpg.rf.c7d17d0712032b8ace6806b9362b377d.jpg
104_jpg.rf.6ba722a47b3bfc9abf6a7d59b4f75af8.jpg
107_jpg.rf.ff4a3d0d3eb6bb27b614d8a0b86ac03b.jpg
17_jpg.rf.5ecad843d9035d4b0b838e18e81e5bd6.jpg
21_jpg.rf.e8cb8f4d57a0343ae2545124ee3f95ea.jpg
25_jpg.rf.549e778dc23049bb90e1d628578bdbdb.jpg
28_jpg.rf.22e4787b575efef2b6f3fb05c8115d8c.jpg
3_jpg.rf.7909c937fc0c81223b0376a139397a56.jpg
53_jpg.rf.8f67477c98b984c54a0996e86c14beaf.jpg
54_jpg.rf.954c64b408a68b418b5d8216af55d1e1.jpg
62_jpg.rf.704793db260e37ca0769c77e971bd2a4.jpg
74_jpg.rf.13cde28d7c32faa556bacfd884302eeb.jpg
97_jpg.rf.2f6a37b05ded11efa3c3f7ae69e294f4.jpg
9_jpg.rf.2ff7ffb825a4648ef6c03136a74b9f39.jpg
IMG-20251121-WA0022_jpg.rf.e084689b77cb750ecc6ddb30075b9855.jpg
IMG-20251121-WA0024_jpg.rf.a5d5b9a934e4eeedfa058a8ae7e81705.jpg
IMG-20251121-WA0028_jpg.rf.d02183ce9b636

In [26]:
!cp -r /content/datasets/Packaging-4 /content/drive/MyDrive/

In [27]:
!ls /content/drive/MyDrive/Packaging-4

data.yaml  README.roboflow.txt	test  train  valid


In [28]:
!cat /content/datasets/Packaging-4/data.yaml

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 9
names: ['carton_box', 'export_mark_label', 'fragile_label', 'handle_with_care_label', 'keep_dry_label', 'package_damage', 'protect_from_heat_label', 'this_side_up_label', 'wooden_box']

roboflow:
  workspace: ishadis-workspace-52e3q
  project: packaging-8htcz
  version: 4
  license: Private
  url: https://app.roboflow.com/ishadis-workspace-52e3q/packaging-8htcz/4

In [29]:
!find /content -name "best.pt"

/content/datasets/runs/detect/train/weights/best.pt


In [30]:
!mkdir -p /content/drive/MyDrive/Package-4/model
!cp /content/datasets/runs/detect/train/weights/best.pt /content/drive/MyDrive/Package-4/model/

In [31]:
!ls /content/drive/MyDrive/Package-4/model

best.pt


In [32]:
!du -h /content/drive/MyDrive/Package-4/model/best.pt

19M	/content/drive/MyDrive/Package-4/model/best.pt


In [33]:
!ls /content/drive/MyDrive/Package-4

dataset  model


In [34]:
%cd /content/drive/MyDrive/Package-4

/content/drive/MyDrive/Package-4


In [35]:
!git init
!git add .
!git commit -m "Initial commit - Package Damage Detection using YOLOv11"

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/Package-4/.git/
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@a3af30638211.(none)')


In [36]:
!git config --global user.name "ishadiumeshika"
!git config --global user.email "ishadiiumeshika@gmail.com"

In [37]:
!git add .
!git commit -m "Initial commit - Package Damage Detection using YOLOv11"

[master (root-commit) 8ae4ba8] Initial commit - Package Damage Detection using YOLOv11
 2 files changed, 7 insertions(+)
 create mode 100644 dataset/data.yaml
 create mode 100644 model/best.pt


In [38]:
!ls

dataset  model


In [39]:
%%writefile README.md
# Package Damage Detection using YOLOv11

This project uses YOLOv11 to detect package damage and shipping labels from package images.

## Classes
- carton_box
- export_mark_label
- fragile_label
- handle_with_care_label
- keep_dry_label
- package_damage
- protect_from_heat_label
- this_side_up_label
- wooden_box

## Model
Custom-trained YOLOv11 model on a Roboflow dataset.

Writing README.md


In [40]:
!git add .
!git commit -m "Add dataset, model, and README"

[master 0643e2e] Add dataset, model, and README
 1 file changed, 17 insertions(+)
 create mode 100644 README.md


In [41]:
!git branch -M main
!git remote add origin https://github.com/ishadiumeshika/Research_yolo-damage-detection.git

In [42]:
!git push -u origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets

from google.colab import userdata
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Sc378P9HEa3kqQL6xWWH")
project = rf.workspace("ishadis-workspace-52e3q").project("packaging-8htcz")
version = project.version(4)
dataset = version.download("yolov11")


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
data="/content/datasets/Packaging-4/data.yaml",
epochs=100,
imgsz=640,
batch=16,
device=0
)


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data="/content/datasets/Packaging-4/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0
)

In [ ]:
!pip install ultralytics


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
data="/content/datasets/Packaging-4/data.yaml",
epochs=100,
imgsz=640,
batch=16,
device=0
)


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/datasets/runs/detect/train/weights/best.pt")

results = model.predict(
    source="/content/datasets/Packaging-4/test/images",
    conf=0.25,
    save=True
)

In [ ]:
/content/datasets/runs/detect/train/weights/best.pt

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/datasets/runs/detect/train/weights/best.pt")


In [ ]:
from google.colab import files

files.download('/content/datasets/runs/detect/train/weights/best.pt')


In [ ]:
!ls /content/datasets/runs/detect/train/weights

In [ ]:
!ls /content

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/datasets/runs/detect/train/weights/best.pt")

results = model.predict(
    source="/content/360_video.mp4",
    conf=0.25,
    save=True
)

In [ ]:
!ls /content/datasets/runs/detect/predict-2

In [ ]:
!ls /content/datasets

In [ ]:
!pip install roboflow

In [ ]:
from IPython.display import Video

Video("/content/datasets/runs/detect/predict-2/360_video.mp4", embed=True)

In [ ]:
!ls /content

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/datasets/runs/detect/train/weights/best.pt")

results = model.predict(
    source="/content/abc.mp4",
    conf=0.25,
    save=True
)

In [ ]:
!ls /content

In [ ]:
!ls /content

In [ ]:
Results saved to /content/datasets/runs/detect/predict-2

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/datasets/runs/detect/train/weights/best.pt")

results = model.predict(
    source="/content/abc.mp4",
    conf=0.25,
    save=True
)

In [ ]:
!ls /content/datasets/runs/detect

In [ ]:
!ls /content/datasets/runs/detect/predict2

In [ ]:
!ls /content/datasets/runs/detect

In [ ]:
!ls /content/datasets/runs/detect/predict-4

In [ ]:
from IPython.display import Video

Video("/content/datasets/runs/detect/predict-4/abc.mp4", embed=True)

In [ ]:
from google.colab import files

files.download("/content/datasets/runs/detect/predict-4/abc.mp4")

In [ ]:
!ls -l /content/datasets/runs/detect/predict-4

In [ ]:
from google.colab import files

files.download("/content/datasets/runs/detect/predict-4/abc.avi")

In [ ]:
!ls -l /content/datasets/runs/detect/predict
!ls -l /content/datasets/runs/detect/predict-2
!ls -l /content/datasets/runs/detect/predict-3

In [ ]:
from google.colab import files

files.download("/content/datasets/runs/detect/predict-2/360_video.avi")

In [ ]:
!ls /content

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/datasets/runs/detect/train/weights/best.pt")

results = model.predict(
    source="/content/damaged.mp4",
    conf=0.25,
    save=True
)

In [ ]:
!ls /content/datasets/runs/detect/predict-2

In [ ]:
from IPython.display import Video

Video("/content/datasets/runs/detect/predict-2/damaged.mp4", embed=True)

In [ ]:
from google.colab import files

files.download("/content/datasets/runs/detect/predict-4/damaged.mp4")

In [ ]:
from google.colab import files

files.download("/content/datasets/runs/detect/predict-4/damaged.avi")

In [ ]:
!ls -l /content/datasets/runs/detect/predict-4

In [ ]:
!ls -l /content/datasets/runs/detect/predict-5

In [ ]:
from google.colab import files

files.download("/content/datasets/runs/detect/predict-4/damaged.avi")

In [ ]:
!ls /content/datasets/runs/detect/

In [ ]:
!find /content -type f \( -name "*.avi" -o -name "*.mp4" \)

## Custom Training

In [ ]:
from google.colab import files
files.download("/content/datasets/runs/detect/predict-5/damaged.avi")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolo11s.pt data={dataset.location}/data.yaml epochs=10 imgsz=640 plots=True

**NOTE:** The results of the completed training are saved in `{HOME}/runs/detect/train/`. Let's examine them.

In [ ]:
!ls {HOME}/runs/detect/train/

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/train/results.png', width=600)

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/train/val_batch0_pred.jpg', width=600)

## Validate fine-tuned model

In [ ]:
!yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

## Inference with custom model

In [ ]:
!yolo task=detect mode=predict model={HOME}/runs/detect/train/weights/best.pt conf=0.25 source={dataset.location}/test/images save=True

**NOTE:** Let's take a look at few results.

In [ ]:
import glob
import os
from IPython.display import Image as IPyImage, display

latest_folder = max(glob.glob(f'{HOME}/runs/detect/predict*/'), key=os.path.getmtime)
for img in glob.glob(f'{latest_folder}/*.jpg')[:3]:
    display(IPyImage(filename=img, width=600))
    print("\n")

## Deploy model on Roboflow

Once you have finished training your YOLOv11 model, you’ll have a set of trained weights ready for use. These weights will be in the `/runs/detect/train/weights/best.pt` folder of your project. You can upload your model weights to Roboflow Deploy to use your trained weights on our infinitely scalable infrastructure.

The `.deploy()` function in the [Roboflow pip package](https://docs.roboflow.com/python) now supports uploading YOLOv11 weights.

In [ ]:
project.version(dataset.version).deploy(model_type="yolov11", model_path=f"{HOME}/runs/detect/train/")

In [ ]:
!pip install inference

In [ ]:
import os, random, cv2
import supervision as sv
import IPython
import inference

model_id = project.id.split("/")[1] + "/" + dataset.version
model = inference.get_model(model_id, userdata.get('ROBOFLOW_API_KEY'))

# Location of test set images
test_set_loc = dataset.location + "/test/images/"
test_images = os.listdir(test_set_loc)

# Run inference on 4 random test images, or fewer if fewer images are available
for img_name in random.sample(test_images, min(4, len(test_images))):
    print("Running inference on " + img_name)

    # Load image
    image = cv2.imread(os.path.join(test_set_loc, img_name))

    # Perform inference
    results = model.infer(image, confidence=0.4, overlap=30)[0]
    detections = sv.Detections.from_inference(results)

    # Annotate boxes and labels
    box_annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()
    annotated_image = box_annotator.annotate(scene=image, detections=detections)
    annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

    # Display annotated image
    _, ret = cv2.imencode('.jpg', annotated_image)
    i = IPython.display.Image(data=ret)
    IPython.display.display(i)


## 🏆 Congratulations

### Learning Resources

Roboflow has produced many resources that you may find interesting as you advance your knowledge of computer vision:

- [Roboflow Notebooks](https://github.com/roboflow/notebooks): A repository of over 20 notebooks that walk through how to train custom models with a range of model types, from YOLOv7 to SegFormer.
- [Roboflow YouTube](https://www.youtube.com/c/Roboflow): Our library of videos featuring deep dives into the latest in computer vision, detailed tutorials that accompany our notebooks, and more.
- [Roboflow Discuss](https://discuss.roboflow.com/): Have a question about how to do something on Roboflow? Ask your question on our discussion forum.
- [Roboflow Models](https://roboflow.com): Learn about state-of-the-art models and their performance. Find links and tutorials to guide your learning.

### Convert data formats

Roboflow provides free utilities to convert data between dozens of popular computer vision formats. Check out [Roboflow Formats](https://roboflow.com/formats) to find tutorials on how to convert data between formats in a few clicks.

### Connect computer vision to your project logic

[Roboflow Templates](https://roboflow.com/templates) is a public gallery of code snippets that you can use to connect computer vision to your project logic. Code snippets range from sending emails after inference to measuring object distance between detections.